# Feature Extraction — BISINDO

## Pipeline
```
frames.npy (dari PreProcessing)
  → [Step 1] KeypointExtractor (MediaPipe) → (T, 126) per video → *.npy
  → [Step 2] SequenceBuilder              → (N, 30, 126) split  → .npz
```

Jalankan **Step 1 dulu, baru Step 2**.

In [ ]:
import sys, os
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
ROOT = NOTEBOOK_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'ROOT project: {ROOT}')

In [ ]:
from FeatureExtraction.Config.feature_config import *
create_output_dirs()

---
## Step 1 — Keypoint Extraction (MediaPipe)

MediaPipe otomatis:
1. Deteksi area tangan di frame (Palm Detector)
2. Ekstrak 21 keypoint dari area itu (Hand Landmark)

Output per video: `(T, 126)` — 2 tangan × 21 keypoint × 3 coords

> ⏱️ **Estimasi waktu:** ~30–60 menit untuk 1600 video

In [ ]:
from FeatureExtraction.Scripts.keypoint_extractor import KeypointExtractor

extractor = KeypointExtractor()
extractor.process_all(skip_existing=True)
extractor.close()

### Verifikasi Step 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from FeatureExtraction.Config.feature_config import get_keypoint_path

test_signer, test_label, test_sample = 0, 0, 1
vid_name = f'signer{test_signer}_label{test_label}_sample{test_sample}'
kpt_path = get_keypoint_path(test_signer, test_label, test_sample)

kpts = np.load(kpt_path)  # (T, 126)
print(f'Shape     : {kpts.shape}')
print(f'Min/Max   : {kpts.min():.4f} / {kpts.max():.4f}')

detected = np.any(kpts != 0, axis=1)
print(f'Terdeteksi: {detected.sum()}/{len(kpts)} frame ({detected.mean()*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(kpts[:, 0],  label='wrist kiri  (x)')
ax.plot(kpts[:, 63], label='wrist kanan (x)')
ax.set_title(f'Keypoint wrist x-coord — {vid_name}')
ax.set_xlabel('Frame')
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 2 — Sequence Builder

- Pad/crop temporal ke 30 frame
- Stratified split → train / val / test
- Simpan `.npz`

> ⏱️ **Estimasi waktu:** ~1–5 menit

In [ ]:
from FeatureExtraction.Scripts.sequence_builder import SequenceBuilder, verify_sequences

builder = SequenceBuilder()
builder.build(skip_existing=False)

In [ ]:
verify_sequences()

### Distribusi label per split

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from FeatureExtraction.Config.feature_config import get_sequence_output_path, NUM_LABELS

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, split in zip(axes, ['train', 'val', 'test']):
    data   = np.load(get_sequence_output_path(split), allow_pickle=True)
    counts = np.bincount(data['y'], minlength=NUM_LABELS)
    ax.bar(range(NUM_LABELS), counts, color='steelblue', alpha=0.8)
    ax.set_title(f'{split} — {len(data["y"])} sequences')
    ax.set_xlabel('Label Index')
    ax.set_ylabel('Jumlah')
plt.suptitle('Distribusi Label per Split')
plt.tight_layout()
plt.show()

---
## ✅ Output Akhir

```
Datasets/WLBisindo_Features/
  keypoints/              ← *.npy per video (T, 126)
  sequences/
    train_sequences.npz   ← X: (N_train, 30, 126), y: (N_train,)
    val_sequences.npz
    test_sequences.npz
```

### Load di Training/
```python
import numpy as np
data = np.load('Datasets/WLBisindo_Features/sequences/train_sequences.npz')
X = data['X']  # (N, 30, 126) float32
y = data['y']  # (N,)         int64
```